# Live Training Dashboard
Run the last cell — it refreshes every 30 s until you interrupt it (Kernel ▸ Interrupt).
Shows: GPU utilization, batch-level progress of every active run, epoch AUC curves for all model generations.

In [ ]:
from pathlib import Path
import re, time, subprocess
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output

L = Path("/work/07880/devansh/lonestar/rsna_knee/logs")
RUNS = {  # display name -> log file
    **{f"r1-convnext f{f}": L/f"train_fold{f}.log" for f in range(4)},
    **{f"r1-effnetv2 f{f}": L/f"train_v2_fold{f}.log" for f in range(4)},
    **{f"r2-convnext f{f}": L/f"train_r2_v1_fold{f}.log" for f in range(4)},
    **{f"r2-effnetv2 f{f}": L/f"train_r2_v2_fold{f}.log" for f in range(4)},
    **{f"r2-seresnext f{f}": L/f"train_r2_v3_fold{f}.log" for f in range(4)},
    **{f"r3-hires256 f{f}": L/f"train_r3_hires_fold{f}.log" for f in range(4)},
}
EP = re.compile(r"ep(\d+): loss ([\d.]+) \| val-teacher AUC ([\d.]+) \| gold AUC ([\d.]+)")
BATCH = re.compile(r"ep(\d+) (\d+)/(\d+) loss ([\d.]+)")

def parse(p):
    if not p.exists(): return None, None, None
    txt = p.read_text(errors="ignore")
    eps = pd.DataFrame(EP.findall(txt), columns=["ep","loss","val","gold"]).astype(float)
    bs = BATCH.findall(txt)
    last_batch = bs[-1] if bs else None
    done = "DONE fold" in txt
    return eps, last_batch, done

def gpu_status():
    try:
        out = subprocess.run(["nvidia-smi","--query-gpu=index,utilization.gpu,memory.used,memory.total",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10).stdout
        return [l.split(", ") for l in out.strip().splitlines()]
    except Exception as e:
        return []
print("helpers ready — run the next cell for the live view")

In [ ]:
while True:
    clear_output(wait=True)
    print(time.strftime("%H:%M:%S"), "— live training dashboard (interrupt kernel to stop)\n")
    for g in gpu_status():
        bar = "#" * int(int(g[1]) / 5)
        print(f"GPU {g[0]}: [{bar:<20}] {g[1]:>3}% util  {int(g[2])/1024:.1f}/{int(g[3])/1024:.0f} GB")
    print()
    active, curves = [], {}
    for name, p in RUNS.items():
        eps, lb, done = parse(p)
        if eps is None: continue
        if len(eps): curves[name] = eps
        if done:
            status = f"DONE  (best gold {eps.gold.max():.3f})" if len(eps) else "DONE"
        elif lb:
            ep, i, n, loss = lb
            pct = 100 * int(i) / int(n)
            status = f"ep{ep} [{int(pct)//5*'#':<20}] {pct:4.1f}%  loss {loss}"
            active.append(name)
        else:
            status = "starting..."
        marker = ">>" if name in active else "  "
        print(f"{marker} {name:<18} {status}")
    if curves:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        for name, h in curves.items():
            r2 = name.startswith(("r2", "r3"))
            style = dict(marker="o", ms=3, alpha=0.9 if r2 else 0.35,
                         lw=2 if r2 else 1, ls="-" if r2 else "--")
            axes[0].plot(h.ep, h.gold, label=name if r2 else None, **style)
            axes[1].plot(h.ep, h.loss, label=name if r2 else None, **style)
        axes[0].set_title("gold-58 AUC by epoch (solid=round2, faded=round1)"); axes[0].grid(alpha=.3)
        axes[1].set_title("train loss"); axes[1].grid(alpha=.3)
        axes[0].legend(fontsize=7, ncol=2); axes[1].legend(fontsize=7, ncol=2)
        plt.tight_layout(); plt.show()
    time.sleep(30)